In [1]:
from IPython.display import HTML
with open("custom/style.css", "r") as f:
    custom_style = f.read()
hide_cell_style = "<style>.jp-Cell:has(.hide-me) { display: yes; }</style>"
HTML(f"<div class='hide-me'></div><style>{custom_style}</style>{hide_cell_style}")

<h1><center><font color='blue'>Introduction to Parallel Programming with MPI<br><br>Blocking Collective Communication</font></center></h1>

# <font color='blue'>Blocking Collective Communication</font>

## Collectives in MPI

Collectives: operations <font color='green'>including all ranks</font> of a communicator

<font color='green'>All ranks must call the function!</font>

- <font color='green'>Blocking</font> variants: buffer can be reused after return
- <font color='green'>Nonblocking</font> variants (since MPI 3.0):
  buffer can be used after completion (<font color='green'>MPI_Wait*/MPI_Test*</font>)

- May or may not synchronize the processes
- <font color='red'>Cannot interfere with point-to-point communication</font>
  - <font color='green'>Completely separate modes of operation!</font>

- <font color='green'>Rules</font> for all collectives
  - Data type matching
  - No tags
  - Count must be exact, i.e., there is only one message length, buffer must be large enough
- Types:
  - <font color='green'>Synchronization</font> (barrier)
  - <font color='green'>Data movement</font> (broadcast, scatter, gather, all-to-all)
  - Collective <font color='green'>computation</font> (reduction, scan)
  - <font color='green'>Combinations</font> of data movement and computation (reduction + broadcast)
- General assumption: <font color='green'>MPI does a better job</font> at collectives <font color='green'>than you</font> trying to emulate them with a collection of point-to-point calls

## Barrier

<img src="figs/barrier.svg" alt="barrier.svg" style="float: right; margin-right: 15px;" width="25%">
<!--```{image} figs/barrier.svg
:width: 200px
```-->

- Explicit synchronization of all ranks from specified communicator
```cpp
int MPI_Barrier(MPI_Comm comm)
```
- Ranks only return from call after <font color='green'>every rank</font> has called the function
- MPI_Barrier: rarely needed
  - Debugging

## Broadcast

- Sending buffer contents from one rank (`root`) to all ranks

<img src="figs/broadcast.svg" alt="broadcast.svg" width="90%">
<!--```{image} figs/broadcast.svg
:width: 1000px
```-->

```cpp
int MPI_Bcast(void *buffer, int count, MPI_Datatype datatype,
              int root, MPI_Comm comm)
```
- `root` and `comm` must be the same on all processes
- No restrictions on which rank is `root` - often rank 0

## Scatter

- Sending every i-th chunk of an array to the i-th rank:

<img src="figs/scatter.svg" alt="scatter.svg" width="90%">
<!--```{image} figs/scatter.svg
:width: 1000px
```-->

```cpp
int MPI_Scatter(const void *sendbuf, int sendcount, MPI_Datatype sendtype,
                      void *recvbuf, int recvcount, MPI_Datatype recvtype,
                      int root, MPI_Comm comm)
```
- `root` and `comm` must be the same on all processes
- <font color='green'>Type signature</font> of send and receive variables must match
- Usually, `sendcount = recvcount` because `sendtype = recvtype`
  - It is the length of the chunk
- `sendbuf` is ignored on <font color='green'>non-root ranks</font> because there is nothing to send

## Gather

- Receive a message from each rank and place i-th rank's message at i-th position in receive buffer:

<img src="figs/gather.svg" alt="gather.svg" width="90%">
<!--```{image} figs/gather.svg
:width: 1100px
```-->

```cpp
int MPI_Gather(const void *sendbuf, int sendcount, MPI_Datatype sendtype,
                     void *recvbuf, int recvcount, MPI_Datatype recvtype,
                     int root, MPI_Comm comm)
```
- `root` and `comm` must be the same on all processes
- <font color='green'>Type signature</font> of send and receive variables must match
- Usually, `sendcount = recvcount` because `sendtype = recvtype`
- `recvbuf` is ignored on <font color='green'>non-root ranks</font> because there is nothing to receive

## Scatterv: a more flexible scatter

- Send chunks of different sizes to different ranks

<img src="figs/scatterv.svg" alt="scatterv.svg" width="90%">
<!--```{image} figs/scatterv.svg
:width: 1100px
```-->

```cpp
int MPI_Scatterv(const void *sendbuf, const int sendcounts[], const int displs[], MPI_Datatype sendtype,
                       void *recvbuf, int recvcount, MPI_Datatype recvtype,
                       int root, MPI_Comm comm)
```
- `sendcounts[]`: array specifying the number of elements to send to each rank: send `sendcounts[i]` elements to rank `i`
- `displs[]`:   integer array specifying the displacements in `sendbuf` from which to take the outgoing data to           each rank, specified in number of elements

## Gatherv: a more flexible gather

- Receive segments of different sizes from different ranks
```cpp
int MPI_Gatherv(const void *sendbuf, int sendcount, MPI_Datatype sendtype,
                      void *recvbuf, const int recvcounts[], const int displs[], MPI_Datatype recvtype,
                      int root, MPI_Comm comm)
```
- `recvcounts[]`: array specifying the number of elements to receive from each rank: receive `recvcounts[i]` elements from rank `i`
- `displs[]`: integer array specifying the displacements where received data from specific rank is put in `recvbuf`,            in units of elements

## Allgather

- Combination of <font color='green'>gather</font> and <font color='green'>broadcast</font>

<img src="figs/allgather.svg" alt="allgather.svg" width="90%">
<!--```{image} figs/allgather.svg
:width: 1100px
```-->

<div style="text-align: center; font-size: 24px; max-width: 2000px;">
    <span">
        In this example: <code>sendcount=recvcount=2</code>
    </span>
</div>

```cpp
int MPI_Allgather(const void *sendbuf, int  sendcount, MPI_Datatype sendtype,
                        void *recvbuf, int recvcount, MPI_Datatype recvtype, MPI_Comm comm)
```
- Also available: `MPI_Allgatherv()` (cf. `MPI_Gatherv()`)
- Why not just use gather followed by a broadcast instead?
  - MPI library has <font color='green'>more options for optimization</font>
  - General assumption: Combined collectives are faster than using separate ones

## Alltoall

- Each rank: send i-th chunk to i-th rank

<img src="figs/alltoall.svg" alt="alltoall.svg" width="90%">
<!--```{image} figs/alltoall.svg
:width: 1100px
```-->

```cpp
int MPI_Alltoall(const void *sendbuf, int sendcount, MPI_Datatype sendtype,
                       void *recvbuf, int recvcount, MPI_Datatype recvtype, MPI_Comm comm)
```

- More flexible variants:
  - <font color='green'>MPI_Alltoallv</font>: Allows different number of elements to be send/received by each rank
  - <font color='green'>MPI_Alltoallw</font>: Allows also different data types and displacements in bytes

## Summary of MPI Collective Communications

- MPI (blocking) <font color='green'>collectives</font>
  - <font color='green'>All ranks</font> in communicator <font color='green'>must call</font> the function <p></p>

- <font color='green'>Communication</font> and <font color='green'>synchronization</font>
  - Barrier, broadcast, scatter, gather, and combinations thereof <p></p>

- <font color='green'>In-place buffer</font> specification `MPI_IN_PLACE`
  - Save some space if needed

## Quiz:

1. Why should one use collective communication rather than emulating by a set of point-to-point calls? <br>
   <details style="display: inline;">
       <summary style="display: inline; cursor: pointer;">
           <b>Answer: </b>
       </summary> Implementations are optimized for efficiency, and are also adapted to the hardware environment at hand (not guaranteed, however). You don't want to redo the work of countless MPI researchers.
   </details> <p></p>

2. Can MPI collective communications <font color='green'>interfere</font> with point-to-point calls?
   <ol style="list-style-type: lower-alpha;">
   <li>Yes</li>
   <li>No</li>
   </ol>
   <details style="display: inline;">
       <summary style="display: inline; cursor: pointer;">
           <b>Answer: </b>
       </summary> b., because they are completely separate modes of operation.
   </details> <p></p>
  
3. For a collective communication, it is <font color='green'>not necessary</font> every process of a communicator to call it?
   <ol style="list-style-type: lower-alpha;">
   <li>Correct</li>
   <li>Incorrect</li>
   </ol>
   <details style="display: inline;">
       <summary style="display: inline; cursor: pointer;">
           <b>Answer: </b>
       </summary> b., it is necessary.
   </details> <p></p>

4. To send an <font color='green'>identical piece of data</font> to all other processes in a communicator, which collective call should be used?
   <ol style="list-style-type: lower-alpha;">
   <li>MPI_Gather</li>
   <li>MPI_Bcast</li>
   <li>MPI_Scatter</li>
   <li>MPI_Alltoall</li>
   </ol>
   <details style="display: inline;">
       <summary style="display: inline; cursor: pointer;">
           <b>Answer: </b>
       </summary> b.
   </details> <p></p>

5. Which of the following collective calls is similar to the process of <font color='green'>transposing a matrix</font> in mathematics?
   <ol style="list-style-type: lower-alpha;">
   <li>MPI_Gather</li>
   <li>MPI_Bcast</li>
   <li>MPI_Scatter</li>
   <li>MPI_Alltoall</li>
   </ol>
   <details style="display: inline;">
       <summary style="display: inline; cursor: pointer;">
           <b>Answer: </b>
       </summary> d.
   </details>